# 课程资料问答助手：RAG 检索与重排序效果对比

**选题 B，可运行本地模型版本。** 四位成员的分工和讲稿见配套材料。

知识库为 20 篇自编教学资料，测试题为 12 道内部构造题，结果不代表外部盲测。模型与参数完整记录在 `results/summary.json`。本 Notebook 同时包含理论、代码说明、已保存结果与现场问答入口。

## 1. 理论与流程

离线：读取资料 → 字符切分 → BGE 向量化 → 持久化向量与来源。
在线：编码问题 → 召回 10 个 → 原始顺序或 Cross-Encoder 重排 → 选 3 个 → 同一 Qwen 模型生成。

RAG 利用外部资料提供依据，微调改变模型参数，二者可以结合。RAG 仍可能因检索失败或生成误读而出错。Chunk 太小容易切断证据，太大可能带入干扰；overlap 缓解边界问题，也会增加冗余。

BGE 使用 CLS 池化和 L2 归一化，归一化后点积等于余弦相似度。Bi-Encoder 独立编码便于缓存，Cross-Encoder 联合编码文本对，适合精排候选。余弦分数与重排分数不能直接比较大小。

In [1]:
from pathlib import Path
import json, sys
from IPython.display import display, Markdown
ROOT = Path.cwd()
if not (ROOT / "rag_engine.py").exists():
    if (ROOT / "RAG课程作业交付" / "rag_engine.py").exists():
        ROOT = ROOT / "RAG课程作业交付"
    else:
        raise RuntimeError("请从交付文件夹打开 Notebook")
sys.path.insert(0, str(ROOT))
from rag_engine import RAG, CONFIG, MODELS, corpus_chunks, split_text
print("Python:", sys.executable)
print("模型:", json.dumps(MODELS, ensure_ascii=False, indent=2))
print("配置:", CONFIG)

<WORKSPACE>\.runtime\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python: <WORKSPACE>\.runtime\Scripts\python.exe
模型: {
  "embedding": "BAAI/bge-small-zh-v1.5",
  "reranker": "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1",
  "generator": "Qwen/Qwen2.5-0.5B-Instruct"
}
配置: {'chunk_size': 180, 'overlap': 36, 'top_k': 10, 'context_k': 3, 'max_new_tokens': 128, 'seed': 42, 'threads': 4}


## 2. 资料与切分

正文字符位置可回到原始资料核验。这里的 180/36 指字符，不是 token。参考答案保存在独立测试文件，不进入检索、重排或生成。

In [2]:
chunks = corpus_chunks()
print("文档数:", len({c["doc_id"] for c in chunks}), "片段数:", len(chunks))
for c in chunks[:2]:
    print(c["id"], "正文字符范围:", (c["start"], c["end"]))
    print(c["text"], "\n")
example = list(split_text("甲乙丙丁戊己庚辛壬癸", size=6, overlap=2))
print("切分边界示例:", example)

文档数: 20 片段数: 37
D01-C01 正文字符范围: (0, 180)
RAG 将外部知识检索与语言模型生成结合。离线阶段读取文档、清洗、切分、计算片段向量并持久化。在线阶段编码问题、召回片段、组织上下文，再由语言模型生成答案。知识库更新后需要更新相应索引，但通常不必重新训练生成模型。回答应提供能定位到原文的来源。

微调通过训练改变模型参数，可用于风格、任务格式或能力适配。RAG 主要让模型利用外部证据，两者可以结合。RAG  

D01-C02 正文字符范围: (144, 220)
务格式或能力适配。RAG 主要让模型利用外部证据，两者可以结合。RAG 不保证完全消除幻觉：召回错误、证据缺失、上下文冲突和生成误读都可能导致回答错误。 

切分边界示例: [(0, 6, '甲乙丙丁戊己'), (4, 10, '戊己庚辛壬癸')]


## 3. 加载本地模型并预热

首次运行本单元格需要等待。模型准备好以后，台上只运行下面的查询单元格。向量扫描采用 NumPy 精确检索，并未部署 Chroma 或 Milvus。所有模型强制从本地目录加载。

In [3]:
rag = RAG()
_ = rag.compare("请介绍本知识库的资料范围。", generate=False)
print("模型已加载，向量矩阵形状:", rag.vectors.shape)
print("可以执行现场查询。")

模型已加载，向量矩阵形状: (37, 512)
可以执行现场查询。


## 4. 现场问答：修改 question 后运行

输出前 10 个候选的排序、最终上下文、两组生成回答和计时。两个分数只在各自模型内部用于排序。资料外问题也可以直接输入，观察小模型能否说明资料不足。

### 补充排序演示：原第 4 名提升到第 1 名

从 12 条口语化探索查询中选择的实际案例，仅解释排序机制，不计入固定 12 题的实验统计。

In [4]:
ranking_question = '搜到的材料来自相关文件，为什么仍不等于能答完整？'
ranking_demo = rag.compare(ranking_question, generate=False)
print("问题:", ranking_question)
for i, (a,b) in enumerate(zip(ranking_demo["candidates"], ranking_demo["reranked_candidates"]),1):
    print(f"{i}\t向量: {a['id']}\t重排: {b['id']}（原第{b['vector_rank']}名）")
print("重排序第一名原文:", ranking_demo["rerank_context"][0]["text"])

问题: 搜到的材料来自相关文件，为什么仍不等于能答完整？
1	向量: D06-C01	重排: D08-C02（原第4名）
2	向量: D19-C01	重排: D06-C01（原第1名）
3	向量: D06-C02	重排: D17-C01（原第8名）
4	向量: D08-C02	重排: D02-C02（原第10名）
5	向量: D09-C01	重排: D15-C02（原第6名）
6	向量: D15-C02	重排: D19-C01（原第2名）
7	向量: D15-C01	重排: D15-C01（原第7名）
8	向量: D17-C01	重排: D06-C02（原第3名）
9	向量: D08-C01	重排: D09-C01（原第5名）
10	向量: D02-C02	重排: D08-C01（原第9名）
重排序第一名原文: 实。资料外问题没有相关文档，不能混入普通证据命中率的分母。评价前固定参考证据与判定规则，避免看完结果再修改标准。


### 固定题集 Q10 的生成对比

修改下面的问题可继续提问。模型只看到当前资料，不读取参考答案。

In [5]:
question = '为什么修改Embedding模型后，即使向量维度没变也要重新建库？'
demo = rag.compare(question)
print("问题:", question)
print("名次\t向量排序\t\t重排序")
for i, (a,b) in enumerate(zip(demo["candidates"], demo["reranked_candidates"]),1):
    print(f"{i}\t{a['id']} ({a['vector_score']:.3f})\t{b['id']} ({b['rerank_score']:.3f})")
print("\n基础回答:\n", demo["baseline_answer"])
print("\n重排回答:\n", demo["rerank_answer"])
print("\n阶段合计耗时:", round(demo["baseline_total_s"],2), round(demo["rerank_total_s"],2), "秒")

问题: 为什么修改Embedding模型后，即使向量维度没变也要重新建库？
名次	向量排序		重排序
1	D15-C01 (0.706)	D15-C01 (8.388)
2	D10-C01 (0.602)	D10-C01 (-1.845)
3	D11-C02 (0.600)	D01-C01 (-2.281)
4	D02-C02 (0.593)	D02-C02 (-2.867)
5	D19-C01 (0.591)	D03-C01 (-3.408)
6	D11-C01 (0.587)	D11-C01 (-4.431)
7	D01-C01 (0.556)	D11-C02 (-4.435)
8	D18-C02 (0.546)	D19-C01 (-5.715)
9	D19-C02 (0.535)	D19-C02 (-7.706)
10	D03-C01 (0.534)	D18-C02 (-8.248)

基础回答:
 理由：在修改了Embedding模型后，尽管向量维度没有发生变化，但仍然需要重新建立库以确保所有相关组件都已正确配置和更新。这包括更新模型标识、模型文件校验值、归一化参数等，这些都需要经过验证和确认才能进行。因此，在修改Embedding模型后，必须重新构建库以避免重复工作，并确保所有必要的信息都被正确地存储和处理。

重排回答:
 理由：在修改了Embedding模型后，尽管向量维度没有发生变化，但是由于语义空间可能已经变化，因此需要重新建立索引以确保其兼容性。这是因为语义空间的变化可能导致原有的向量表示不再准确地反映了原始文档的内容，从而影响后续的搜索结果准确性。

阶段合计耗时: 13.73 10.26 秒


## 5. 回看原文与实现

引用编号仅是定位方式，不能替代事实核验。下面显示实际送给生成器的片段。原理对应的真实实现来自 `rag_engine.py`，可以直接阅读与修改。

In [6]:
for c in demo["rerank_context"]:
    print(f"[{c['id']}] {c['file']}，正文字符 {c['start']}:{c['end']}")
    print(c["text"], "\n")
import inspect
print(inspect.getsource(RAG.encode))
print(inspect.getsource(RAG.rerank))

[D15-C01] data/corpus/D15.md，正文字符 0:180
资料更新后，已有片段向量不会自动反映新内容。需要识别变动文档，更新或重建相应片段和向量，同时清理过期标识。本项目通过语料和参数的 SHA256 摘要识别索引是否失效，防止修改资料后继续使用旧缓存。

Embedding 模型或切分规则改变后也应重新建库。查询与文档的向量维度相同仍不足以保证兼容，因为语义空间可能已经变化。记录模型标识、模型文件校验值、归一化方 

[D10-C01] data/corpus/D10.md，正文字符 0:180
比较是否启用 Rerank 时，应保持资料、切分、Embedding、候选集合、最终上下文数量、生成模型和提示词一致。两组改变的因素仅为是否执行重排序。固定测试集既记录改善，也记录无变化和退化，不删除不利案例。

模型加载和建库耗时应与普通查询分开记录。先预热，再测量检索、重排序和生成各阶段耗时，明确端到端计时边界。本项目按固定顺序运行，CPU 缓存和系统负 

[D01-C01] data/corpus/D01.md，正文字符 0:180
RAG 将外部知识检索与语言模型生成结合。离线阶段读取文档、清洗、切分、计算片段向量并持久化。在线阶段编码问题、召回片段、组织上下文，再由语言模型生成答案。知识库更新后需要更新相应索引，但通常不必重新训练生成模型。回答应提供能定位到原文的来源。

微调通过训练改变模型参数，可用于风格、任务格式或能力适配。RAG 主要让模型利用外部证据，两者可以结合。RAG  

    @torch.inference_mode()
    def encode(self,texts):
        tokenizer,model=self.models['embedding']; out=[]
        for start in range(0,len(texts),16):
            inputs=tokenizer(texts[start:start+16],padding=True,truncation=True,max_length=384,return_tensors='pt').to(self.device)
            x=model(**inputs).last_hidden

## 6. 固定测试集的已保存实验

固定候选、最终上下文数量、生成器与提示词，仅改变排序方式。以下读取交付时已执行的完整题集，避免台上重新生成 24 条回答。

Hit@3 按预先标注的相关文档计算，MRR@3 取首个相关片段排名倒数；资料外题不计入检索分母。同文档重叠片段不合并，因此文档命中不保证完整证据或答案正确。

In [7]:
summary = json.loads((ROOT/"results/summary.json").read_text(encoding="utf-8"))
records = json.loads((ROOT/"results/raw_results.json").read_text(encoding="utf-8"))
lines = ["|指标|基础 RAG|加入 Rerank|", "|---|---:|---:|"]
for m in ["hit1", "hit3", "mrr3"]:
    lines.append(f"|{m}|{summary['baseline'][m]:.3f}|{summary['rerank'][m]:.3f}|")
lines.append(f"|平均阶段合计秒|{summary['baseline']['mean_total_s']:.2f}|{summary['rerank']['mean_total_s']:.2f}|")
display(Markdown("\n".join(lines)))
for r in records:
    print(r["id"], r["question"], "\n", r["baseline_metrics"], r["rerank_metrics"])
print("环境:", summary["environment"])

|指标|基础 RAG|加入 Rerank|
|---|---:|---:|
|hit1|1.000|1.000|
|hit3|1.000|1.000|
|mrr3|1.000|1.000|
|平均阶段合计秒|7.63|7.71|

Q01 RAG知识库更新以后，一般需要重新训练生成模型吗？ 
 {'hit1': 1, 'hit3': 1, 'mrr3': 1.0} {'hit1': 1, 'hit3': 1, 'mrr3': 1.0}
Q02 片段切得太小和太大分别可能带来什么问题？ 
 {'hit1': 1, 'hit3': 1, 'mrr3': 1.0} {'hit1': 1, 'hit3': 1, 'mrr3': 1.0}
Q03 向量先做L2归一化后，可以怎样计算余弦相似度？ 
 {'hit1': 1, 'hit3': 1, 'mrr3': 1.0} {'hit1': 1, 'hit3': 1, 'mrr3': 1.0}
Q04 Bi-Encoder与Cross-Encoder在输入和用途上有什么区别？ 
 {'hit1': 1, 'hit3': 1, 'mrr3': 1.0} {'hit1': 1, 'hit3': 1, 'mrr3': 1.0}
Q05 正确证据没有进入初次召回候选时，Rerank能补回来吗？ 
 {'hit1': 1, 'hit3': 1, 'mrr3': 1.0} {'hit1': 1, 'hit3': 1, 'mrr3': 1.0}
Q06 只在答案后面加一个来源编号，就能证明回答受到证据支持吗？ 
 {'hit1': 1, 'hit3': 1, 'mrr3': 1.0} {'hit1': 1, 'hit3': 1, 'mrr3': 1.0}
Q07 Hit@3和MRR@3分别衡量什么？ 
 {'hit1': 1, 'hit3': 1, 'mrr3': 1.0} {'hit1': 1, 'hit3': 1, 'mrr3': 1.0}
Q08 资料外问题应不应该计入普通证据命中率的分母？ 
 {'hit1': 1, 'hit3': 1, 'mrr3': 1.0} {'hit1': 1, 'hit3': 1, 'mrr3': 1.0}
Q09 IVF_FLAT中的nlist和nprobe各表示什么？ 
 {'hit1': 1, 'hit3': 1, 'mrr3': 1.0} {'hit1': 1, 'hit3': 1, 'mrr3': 1.0}
Q10 为什么修改Embedding模型后，即使向量维度没变也要重新建库？ 
 {'hit1': 1, 'hit3': 

## 7. 回答、资料外问题与失败案例

助手初评在 `answer_review.json`，需要组员复核，不是独立人工盲评。检索相关性与回答正确性分开检查。

In [8]:
findings = json.loads((ROOT/"results/presentation_evidence.json").read_text(encoding="utf-8"))
for key in ["answer_finding","case_finding","oos_finding","conclusion"]:
    print(findings[key], "\n")
for r in records:
    if not r["gold_docs"]:
        print(r["id"], r["question"])
        print("基础:", r["baseline_answer"])
        print("重排:", r["rerank_answer"], "\n")

两组检索 Hit@3 均为 100%
回答初评满分题数均为 3/10 

Q10 重排后说清了“语义空间变化”。Q06 两组都未正确解释来源编号与证据支持的区别。检索命中与答案质量需要分开检查。 

2 道资料外题中，两组均仅 1 道可靠拒答，考试答案问题出现编造。 

本题集检索指标已达上限，重排没有带来指标提升。生成端仍是主要误差来源。 

Q11 这门课任课教师的私人手机号码是多少？
基础: 现有资料不足，无法回答。
重排: 现有资料不足，无法回答。 

Q12 这门课期末考试的标准答案是什么？
基础: 这门课期末考试的标准答案是通过自我评估得出的，即学生在学习过程中自己总结出的知识点和理解程度。因此，标准答案应该是学生自己的理解和总结，而不是由老师或其他人提供的答案。所以，答案应该是一个学生的自我评估或者自我总结，而不是一个固定的答案。
重排: 这门课期末考试的标准答案是“D08-C01”表中给出的结果，即“Hit@3”的值。 



## 8. 自主思考题与拓展

- **为什么重排不能弥补漏召回？** 它只能处理已进入候选集合的片段。
- **为什么不把资料外题放进命中率分母？** 没有相关文档，检索指标不适用，应单独检查拒答和编造。
- **为什么重排改善排名却可能不改善答案？** 文档级指标并不保证片段完整，生成器也可能误读证据。
- **PDF 表格如何扩展？** 识别表格区域、保留行列单位，生成可检索描述，回答回查原图；本次未实现。
- **Milvus 参数如何理解？** IVF_FLAT 的 nlist 是建库分区数、nprobe 是查询访问分区数；HNSW 使用邻近图，参数不能混用。本次未部署 Milvus。

这些是依据课程主题自主整理的讨论，原课件思考题尚未提供，未冒充教师原题。完整理论、实际问题与来源见 `实验报告.md` 和 `参考资料.md`。

## 9. 复现实验与检查

完整实验会对每题生成两组回答，耗时明显长于单题 Demo。下面默认不运行，以保护已交付结果。

程序测试覆盖切分边界、覆盖与缓存失效。重新实验使用新的结果目录，保留原始数据。

In [9]:
RUN_FULL_EXPERIMENT = False
if RUN_FULL_EXPERIMENT:
    import subprocess
    subprocess.run([sys.executable, str(ROOT/"run_experiment.py"), "--output", "results_rerun"], cwd=ROOT, check=True)
else:
    print("完整实验未重新执行。交付时的实测结果已展示，现场使用第4节单题演示。")
import subprocess
check = subprocess.run([sys.executable, "-m", "unittest", "test_core.py"], cwd=ROOT, capture_output=True, text=True)
print(check.stdout, check.stderr)
assert check.returncode == 0

完整实验未重新执行。交付时的实测结果已展示，现场使用第4节单题演示。


 ...
----------------------------------------------------------------------
Ran 3 tests in 0.005s

OK

